In [45]:
import os 
import pandas as pd


In [9]:
os.chdir("../")

In [46]:
print(os.getcwd())
data = pd.read_csv("artifacts/data_ingestion/housing.csv")



c:\Users\penze\Desktop\MLdev_ops\student_Performance_p1


 this is used to get the information about the data, such as the number of rows, columns, and the data types of each column. This is important because we need to know the data types of each column in order to perform data validation.
 We use the data we see here and check the data stype and then we use this to go into the schema.yaml to configure the files 

In [47]:
#data.info() 
data.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

this is done to see if there are any null values in the total_bedrooms column, 
which is the column that has the most null values. We fill the null values with the median of the total_bedrooms column, 
which is a common practice to handle missing values in numerical columns.
But rember that we will just use simple imputer in our pipeline to handle missing values, this is just for demonstration purposes.

In [48]:
data["total_bedrooms"].fillna(data["total_bedrooms"].median(), inplace=True)


C:\Users\penze\AppData\Local\Temp\ipykernel_60364\1961396678.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["total_bedrooms"].fillna(data["total_bedrooms"].median(), inplace=True)


In [49]:
data.shape

(20640, 10)

In [51]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class DataValidationConfig:
    root_dir: Path
    status_file: str 
    unzipped_data_dir: Path
    all_schema: dict




In [52]:
from src.student_Performance_p1.constants import *
from src.student_Performance_p1.utils.common import read_yaml, create_directories


In [64]:
class ConfigurationManager:
    def __init__(self):
        # Read config.yaml
        self.config = read_yaml(CONFIG_FILE_PATH)

        # Read params.yaml
        self.params = read_yaml(PARAMS_FILE_PATH)

        # Read schema.yaml
        self.schema = read_yaml(SCHEMA_FILE_PATH)

        # Create main artifacts folder
        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        # Access data_validation section from config.yaml
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        # Create data validation folder
        create_directories([config.root_dir])

        # Create DataValidationConfig object
        data_validation_config = DataValidationConfig(
            root_dir=config.root_dir,
            status_file=config.status_file,
            unzipped_data_dir=config.unzipped_data_dir,
            all_schema=self.schema.COLUMNS
        )

        return data_validation_config

After we read the data we need to compare with the schema and comapre the feature name 

In [65]:
import os 
from src.student_Performance_p1 import logger

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = None
 
            data = pd.read_csv (self.config.unzipped_data_dir)
            all_cols = list(data.columns)
            
            all_schema= self.config.all_schema.keys() # we get the colon names from the schema file, which is a dictionary, so we use the keys() method to get the column names.
            #all_types = self.config.all_schema.values()
 
            validation_status = all(col in all_schema for col in all_cols)
            with open(self.config.status_file, 'w') as f:
                f.write(f'Validación global: {validation_status}\n')
 
            return validation_status  
        except Exception as e:
            logger.exception(f"Error during column validation: {e}")
            raise e


In [66]:
try:   
    config_manager = ConfigurationManager()
    data_validation_config = config_manager.get_data_validation_config()
    data_validator = DataValidation(config=data_validation_config)
    validation_result = data_validator.validate_all_columns()
    print(f"Column validation result: {validation_result}")
except Exception as e:
    logger.exception(f"Error in data validation process: {e}")

[2026-05-26 17:05:34,092: INFO: common: YAML file 'config\config.yaml' Loading successfully.]
[2026-05-26 17:05:34,095: INFO: common: YAML file 'params.yaml' Loading successfully.]
[2026-05-26 17:05:34,099: INFO: common: YAML file 'schema.yaml' Loading successfully.]
[2026-05-26 17:05:34,103: INFO: common: Directories created successfully: ['artifacts']]
[2026-05-26 17:05:34,107: INFO: common: Directories created successfully: ['artifacts/data_validation']]
Column validation result: True
